In [1]:
import os
%load_ext autoreload
%autoreload 2
os.chdir("..")
os.chdir("..")

!dir

 O volume na unidade C � OS
 O N�mero de S�rie do Volume � A2FB-FA56

 Pasta de c:\pyprojects\fast-english

19/05/2025  08:11    <DIR>          .
20/05/2025  07:32    <DIR>          ..
23/04/2025  08:12    <DIR>          .github
14/05/2025  08:15             4.433 .gitignore
23/04/2025  08:36    <DIR>          .venv
19/05/2025  08:06    <DIR>          app
12/05/2025  08:26    <DIR>          database
23/04/2025  08:12    <DIR>          docs
05/05/2025  08:24    <DIR>          extract_data_video
20/05/2025  08:45    <DIR>          logs
28/04/2025  08:06           403.736 poetry.lock
28/04/2025  08:06             1.168 pyproject.toml
23/04/2025  13:07               149 README.md
23/04/2025  13:12               157 requirements.txt
23/04/2025  08:12    <DIR>          tests
12/05/2025  07:58    <DIR>          trash
               5 arquivo(s)        409.643 bytes
              11 pasta(s)   137.260.646.400 bytes dispon�veis


In [29]:
import tkinter as tk
from tkinter import ttk
from app.games.vocabulary_game.main import VocabularyGame
from app.games.game_color.main import ColorGame
from app.games.hangman_game.main import HangmanGame
from app.games.word_shuffle_game.main import WordShuffleGame 
from app.games.text_challenge.main import TextChallengeApp
from app.games.text_reader_app.main import TextReaderApp
from app.utils.data_loader import DataLoader
import json
import os
from pathlib import Path
import numpy as np

from app.stats_words.analyzer import WordStatsAnalyzer, WordLearningAnalyzer

class BasePage(tk.Frame):
    """Classe base para todas as páginas com métodos comuns."""
    def __init__(self, parent, controller, **kwargs):
        super().__init__(parent, controller, **kwargs)
        self.controller = controller

    def centralize_widget(self, widget, rely):
        """Centraliza um widget verticalmente."""
        widget.place(relx=0.5, rely=rely, anchor=tk.CENTER)

    def add_back_button(self, text="Voltar para a Inicial"):
        """Adiciona um botão para voltar à página inicial."""
        back_button = tk.Button(self, text=text, command=lambda: self.controller.show_frame("WordBaseApp"))
        self.centralize_widget(back_button, 0.85)

    def add_stats_view(self):
        pass


class WordBaseApp(BasePage):
    def __init__(self, parent, controller, **kwargs):
        super().__init__(parent, controller, **kwargs)

        self.data_path = os.path.join("database", "extract_data_video", "data", "extracted_data", "{kind}", "data_organize")
        self.save_path_study_word_list = os.path.join("database", "vocabulary", "study_word_list.json")
        self.current_type = "words"

        word_stats_analyzer = WordStatsAnalyzer(list_game_name=["game_data_hangman", "game_data_word_shuffle_game"])
        df = word_stats_analyzer.stats_grouped
        self.word_accuracy_map = dict(zip(df['word'].str.lower(), df['acuracia']))

        self.word_stats_analyzer = WordLearningAnalyzer(word_stats_analyzer.stats_grouped)
        

        self.data = self.create_estructure()

        self.build_ui()

    def create_estructure(self):
        estrutura = {}
        list_kinds = ["words", "phrases"]
        for kind in list_kinds:
            self.loader = DataLoader(base_path=self.data_path.format(kind=kind))
            estrutura[kind] = {}
            for categoria in self.loader.get_categories():
                estrutura[kind][categoria.name] = {}
                for subcat in self.loader.get_subcategories(categoria):
                    estrutura[kind][categoria.name][subcat.name] = []
                    for word_path in self.loader.get_word_paths(subcat):
                        word_path = Path(str(word_path).replace("\\", "/"))
                        estrutura[kind][categoria.name][subcat.name].append(word_path)
        return estrutura

    def build_ui(self):
        # Limpa tudo
        for widget in self.winfo_children():
            widget.destroy()

        # Topo: Botões de tipo
        top_frame = tk.Frame(self)
        top_frame.pack(pady=10)

        words_btn = tk.Button(top_frame, text="Palavras", command=lambda: self.switch_type("words"))
        words_btn.pack(side="left", padx=5)

        phrases_btn = tk.Button(top_frame, text="Frases", command=lambda: self.switch_type("phrases"))
        phrases_btn.pack(side="left", padx=5)

        database_words_btn = tk.Button(top_frame, text="Database", command=lambda: self.switch_type("database"))
        database_words_btn.pack(side="left", padx=5)

        # Área de conteúdo
        self.content_frame = tk.Frame(self)
        self.content_frame.pack(fill="both", expand=True, pady=10)

        self.draw_categories()

    def switch_type(self, type_name):
        self.current_type = type_name

        if type_name == "database":
            # Caminho da pasta com os arquivos
            folder = Path(os.path.join("database", "vocabulary", "save_words"))
            if not folder.exists():
                folder.mkdir(parents=True)

            # Lista arquivos .json
            self.database_files = list(folder.glob("*.json"))
            
            # Chama função que desenha os botões dos arquivos
            self.draw_database_files()
            return

        self.draw_categories()
    
    def draw_database_files(self):
        for widget in self.content_frame.winfo_children():
            widget.destroy()

        if not self.database_files:
            label = tk.Label(self.content_frame, text="Nenhum banco de dados encontrado.")
            label.pack(pady=10)
            return

        label = tk.Label(self.content_frame, text="Escolha um banco de dados:")
        label.pack(pady=10)

        for file_path in self.database_files:
            data = self.load_database_file(file_path)
            btn = tk.Button(
                self.content_frame,
                text=file_path.stem,  # Nome do arquivo sem extensão
                anchor="w",
                # command=lambda path=file_path: self.load_database_file(path)
                command=lambda items=data: self.start_game(words=items)
            )
            btn.pack(fill="x", padx=10, pady=2)

    def load_database_file(self, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Aqui você pode usar os dados como quiser. Exemplo:
        self.loaded_words = data
        return data
        # print(f"Arquivo {path.name} carregado com sucesso!")
        # ou: self.start_game(category="database", subcategory=path.stem, words=data)
    
    def draw_categories(self):
        # TODO: Colocar icones
        for widget in self.content_frame.winfo_children():
            widget.destroy()

        # Scrollable Canvas
        canvas = tk.Canvas(self.content_frame)
        scrollbar = ttk.Scrollbar(self.content_frame, orient="vertical", command=canvas.yview)
        scrollable_frame = tk.Frame(canvas)

        scrollable_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(
                scrollregion=canvas.bbox("all")
            )
        )

        canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)

        canvas.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="both")

        categories = self.data.get(self.current_type, {})

        category_colors = [
            "#FFC1C1", "#C1FFD7", "#C1D4FF", "#FFF5C1", "#E1C1FF", "#C1F0FF",
            "#A8D5BA",  # Verde menta
            "#F9E79F",  # Amarelo claro
            "#AED6F1",  # Azul bebê
            "#F5CBA7",  # Pêssego
            "#D7BDE2",  # Lavanda
            "#FADBD8",  # Rosa claro
            "#D5F5E3",  # Verde claro
            "#FDEBD0",  # Creme
            "#E8DAEF",  # Lilás
            "#F6DDCC",  # Bege
            "#D6EAF8",  # Azul claro
            "#FCF3CF",  # Amarelo pastel
            "#E5E8E8",  # Cinza claro
            "#FDEDEC",  # Rosa muito claro
            "#EBDEF0",  # Roxo claro
        ]

        all_maps, learned_per_category = self.word_stats_analyzer.generate_learning_maps(categories)
        
        for i, (category, subcats) in enumerate(categories.items()):
            category_color = category_colors[i % len(category_colors)]

            # Frame principal da categoria
            cat_outer_frame = tk.Frame(scrollable_frame, bg="white", bd=2, relief="groove")
            cat_outer_frame.pack(fill="x", padx=10, pady=10)

            # Quadrado branco com nome e info da categoria
            cat_info_frame = tk.Frame(cat_outer_frame, bg=category_color, bd=5,)
            cat_info_frame.pack(fill="x", padx=10, pady=5)

            tk.Label(cat_info_frame, text=category, font=("Helvetica", 14, "bold"), bg=category_color, anchor="w").pack(anchor="w")

            total_words = sum(len(words) for words in subcats.values())

            learned_words = learned_per_category.get(category, 0)  # Atualize com os dados reais futuramente
            tk.Label(cat_info_frame, text=f"{learned_words}/{total_words} palavras", bg="white", font=("Helvetica", 10), anchor="w").pack(anchor="w")

            # sub_progress = self.get_accuracy_for_words(word_list)

            # Barra de progresso da categoria
            progress = (learned_words / total_words) * 100 if total_words else 0
            cat_bar = tk.Canvas(cat_info_frame, height=10, bg="#ddd", highlightthickness=0)
            cat_bar.pack(fill="x", pady=(5, 0))
            cat_bar.create_rectangle(0, 0, cat_bar.winfo_reqwidth() * progress / 100, 10, fill="#4caf50", width=0)

            # Subcategorias dentro da categoria (inicialmente ocultas)
            sub_frame = tk.Frame(cat_outer_frame, bg="white")
            # sub_frame.pack(fill="x", padx=10, pady=5)

            def toggle_subcategories(frame=sub_frame, items=subcats, category_name=category):
                if frame.winfo_ismapped():
                    frame.pack_forget()
                else:
                    # Se frame ainda está vazio, cria as subcategorias
                    if not frame.winfo_children():
                        for j, (sub_name, word_list) in enumerate(items.items()):
                            sub_color = "#f9f9f9" if j % 2 == 0 else "#eeeeee"

                            sub_box = tk.Frame(frame, bg=sub_color, bd=1, relief="solid")
                            sub_box.pack(fill="x", pady=5)

                            tk.Label(sub_box, text=sub_name, font=("Helvetica", 12, "bold"),
                                    bg=sub_color, anchor="w").pack(anchor="w", padx=5, pady=(5, 0))
                            
                            # Barra de progresso da subcategoria
                            category_map = all_maps.get(category_name, {})
                            sub_learned = category_map.get(sub_name, {}).get("learned", 0)
                            # print(sub_name, category_map)
                            sub_progress = (sub_learned / len(word_list)) * 100 if word_list else 0

                            tk.Label(sub_box, text=f"{sub_learned}/{len(word_list)} palavras", bg=sub_color,
                                    font=("Helvetica", 10), anchor="w").pack(anchor="w", padx=5)

                            sub_bar = tk.Canvas(sub_box, height=10, bg="#ddd", highlightthickness=0)
                            sub_bar.pack(fill="x", padx=5, pady=(5, 0))
                            sub_bar.create_rectangle(0, 0, sub_bar.winfo_reqwidth() * sub_progress / 100,
                                                    10, fill="#2196f3", width=0)

                            tk.Button(sub_box, text="Estudar",
                                    command=lambda words=word_list: self.start_game(words)
                                    ).pack(pady=5, anchor="e", padx=5)

                    frame.pack(fill="x", padx=10, pady=5)

            # Botão de expandir/retrair subcategorias
            toggle_button = tk.Button(cat_info_frame, text="Mostrar / Ocultar", command=toggle_subcategories)
            toggle_button.pack(anchor="e", pady=(5, 0))


    def start_game(self, words):
        # Limpa conteúdo
        # for widget in self.content_frame.winfo_children():
        #     widget.destroy()
        # Cristiana / cristina Campos
        # Botão voltar
        # back_btn = tk.Button(self, text="Voltar", command=self.build_ui)
        # back_btn.pack(pady=10)

        ## Salvando lista de nomes
        study_word_list = [self.loader._carregar_palavra(path=Path(str(path).replace("\\", "/"))) for path in words]

        # Convert WindowsPath objects to strings in the study_word_list
        serializable_study_word_list = [
            {key: str(value) for key, value in word.items()}
            for word in study_word_list
        ]

        # Save the updated list to the JSON file
        with open(self.save_path_study_word_list, 'w', encoding="utf-8") as json_file:
            json.dump(serializable_study_word_list, json_file, ensure_ascii=False, indent=4)

        self.controller.show_frame("MainPage")


class MainPage(BasePage):
    """Página inicial contendo o menu principal."""

    def __init__(self, parent, controller, **kwargs):
        super().__init__(parent, controller, **kwargs)

        # Título do menu principal
        title_label = tk.Label(self, text="Menu Principal", font=("Arial", 16, "bold"))
        title_label.pack(pady=20)

        # Lista de botões e páginas correspondentes
        games = [
            ("Jogo de Vocabulário", VocabularyGame),
            ("Jogo Da Forca", HangmanGame),
            ("Word Shuffle Game", WordShuffleGame),
            ("Text Reader App", TextReaderApp),
            ("Game Color", ColorGame),
            ("Desafio de Texto", TextChallengeApp),
        ]

        # Criação dinâmica de botões
        for game_name, game_class in games:
            button = tk.Button(self, text=game_name, width=25, height=2,
                               command=lambda g=game_class: controller.show_game_frame(g))
            button.pack(pady=10)

        # Botão adicional
        other_button = tk.Button(self, text="Ir para a Página 2", width=25, height=2,
                                  command=lambda: controller.show_frame("PageTwo"))
        other_button.pack(pady=10)


class GamePage(BasePage):
    """Página genérica para carregar jogos dinamicamente."""

    def __init__(self, parent, controller, game_class, **kwargs):
        super().__init__(parent, controller, **kwargs)
        self.game_instance = None
        self.game_class = game_class

    def load_game(self):
        """Carrega e exibe o jogo apenas quando necessário."""
        if not self.game_instance:
            self.game_instance = self.game_class(self)
            self.game_instance.pack(expand=True, fill="both")
            self.add_back_button()


class PageTwo(BasePage):
    """Página adicional de exemplo."""

    def __init__(self, parent, controller, **kwargs):
        super().__init__(parent, controller, **kwargs)
        label = tk.Label(self, text="Esta é a Página 2", font=("Arial", 14))
        self.centralize_widget(label, 0.4)
        self.add_back_button()

class PageMenuApapter(tk.Frame):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # self.title("Aplicação com Múltiplas Páginas")
        # self.geometry('700x500')

        # Container para armazenar frames
        self.container = tk.Frame(self)
        self.container.pack(fill="both", expand=True)

        # Dicionário para páginas
        self.frames = {}
        self.games = {}

        # Registra a página inicial e a página adicional
        
        # self.register_frame("BaseApp", WordBaseApp)
        self.register_frame("WordBaseApp", WordBaseApp)

        self.register_frame("MainPage", MainPage)
        self.register_frame("PageTwo", PageTwo)

        # Exibe a página inicial
        self.show_frame("WordBaseApp")

    def register_frame(self, name, page_class):
        """Registra um frame."""
        frame = page_class(self.container, self)
        self.frames[name] = frame
        frame.place(relwidth=1, relheight=1)

    def register_game_frame(self, game_class):
        """Registra dinamicamente páginas de jogos."""
        if game_class not in self.games:
            frame = GamePage(self.container, self, game_class)
            self.games[game_class] = frame
            frame.place(relwidth=1, relheight=1)

    def show_frame(self, name):
        """Exibe a página pelo nome."""
        frame = self.frames[name]
        frame.tkraise()
    
    def reset_game(self):
        """Reinicia o jogo atual."""
        if self.game_instance:
            self.game_instance.reset_game()

    def show_game_frame(self, game_class):
        """Exibe uma página de jogo dinamicamente, resetando sempre."""
        if game_class not in self.games:
            self.register_game_frame(game_class)
        else:
            # Destroi o frame anterior e recria do zero
            self.games[game_class].destroy()
            frame = GamePage(self.container, self, game_class)
            self.games[game_class] = frame
            frame.place(relwidth=1, relheight=1)
        
        self.games[game_class].tkraise()
        self.games[game_class].load_game()

# Iniciar app
if __name__ == "__main__":
    try:
        root = tk.Tk()
        game = PageMenuApapter(root)
        game.pack(expand=True, fill="both")
        root.title("PageMenuApapter")
        root.geometry("840x520")
        root.mainloop()

    except Exception as e:
        print(f"Error {e}")
    finally:
        root.destroy()

Não há palavras pendentes no desafio


In [10]:
all_maps = {'ações_1': {'total': 50, 'learned': 16}, 'ações_2': {'total': 41, 'learned': 3}, 'competências': {'total': 25, 'learned': 1}, 'comunicação': {'total': 49, 'learned': 5}, 'eficácia': {'total': 43, 'learned': 3}, 'ensinar_e_aprender': {'total': 34, 'learned': 7}, 'expressão_de_sentimentos': {'total': 22, 'learned': 5}, 'gestão': {'total': 38, 'learned': 2}, 'habilidades_interpessoais': {'total': 50, 'learned': 4}, 'pesquisa': {'total': 24, 'learned': 2}, 'verbos_de_estado': {'total': 20, 'learned': 5}}

all_maps

{'ações_1': {'total': 50, 'learned': 16},
 'ações_2': {'total': 41, 'learned': 3},
 'competências': {'total': 25, 'learned': 1},
 'comunicação': {'total': 49, 'learned': 5},
 'eficácia': {'total': 43, 'learned': 3},
 'ensinar_e_aprender': {'total': 34, 'learned': 7},
 'expressão_de_sentimentos': {'total': 22, 'learned': 5},
 'gestão': {'total': 38, 'learned': 2},
 'habilidades_interpessoais': {'total': 50, 'learned': 4},
 'pesquisa': {'total': 24, 'learned': 2},
 'verbos_de_estado': {'total': 20, 'learned': 5}}

In [ ]:
[info for info in all_maps["ações_1"].values()]

[50, 16]

16

In [ ]:
def create_estructure(data_path):
    estrutura = {}
    list_kinds = ["words", "phrases"]
    for kind in list_kinds:
        loader = DataLoader(base_path=data_path.format(kind=kind))
        estrutura[kind] = {}
        for categoria in loader.get_categories():
            estrutura[kind][categoria.name] = {}
            for subcat in loader.get_subcategories(categoria):
                estrutura[kind][categoria.name][subcat.name] = []
                for word_path in loader.get_word_paths(subcat):
                    word_path = Path(str(word_path).replace("\\", "/"))
                    estrutura[kind][categoria.name][subcat.name].append(word_path)
    return estrutura

data_path = os.path.join("database", "extract_data_video", "data", "extracted_data", "{kind}", "data_organize")

estrutura = create_estructure(data_path)

In [ ]:
from app.stats_words.analyzer import WordStatsAnalyzer, WordLearningAnalyzer
from pathlib import Path
import numpy as np
import pandas as pd
from typing import List, Dict

# Instancia a análise de palavras
word_stats_analyzer = WordStatsAnalyzer(
    list_game_name=["game_data_hangman", "game_data_word_shuffle_game"]
)

# Cria um mapa com a acurácia de cada palavra (em minúsculas)
df = word_stats_analyzer.stats_grouped
word_accuracy_map: Dict[str, float] = dict(zip(df['word'].str.lower(), df['acuracia']))


# Exemplo de uso:
word_stats_analyzer = WordLearningAnalyzer(word_stats_analyzer.stats_grouped)
all_maps, learned_per_category = word_stats_analyzer.generate_learning_maps(estrutura["words"])
print(learned_per_category)


# Chamada final


{'adjetivos': 13, 'advérbios': 29, 'ambiente': 9, 'animais': 7, 'casa': 7, 'cidade': 2, 'cidades': 14, 'comida_e_bebidas': 30, 'compras': 14, 'comunicações': 13, 'corpo': 29, 'cultura': 5, 'descrição_de_pessoas': 13, 'educação': 12, 'esportes': 7, 'moda': 8, 'nomes': 11, 'outras_partes_do_discurso': 44, 'pessoas': 10, 'plantas': 3, 'saude': 4, 'segurança': 6, 'sistemas': 24, 'tempo_de_lazer': 14, 'trabalho': 8, 'transportes': 21, 'verbos': 53}


In [ ]:
subcats

{'descição_de_arte': {'total': 46, 'aprendidas': 0},
 'descrição_de_objetos': {'total': 51, 'aprendidas': 4},
 'descrições_1': {'total': 41, 'aprendidas': 2},
 'descrições_2': {'total': 39, 'aprendidas': 1},
 'descrições_3': {'total': 35, 'aprendidas': 0},
 'pares_1': {'total': 42, 'aprendidas': 5},
 'pares_2': {'total': 40, 'aprendidas': 1},
 'sobre_as_pessoas': {'total': 36, 'aprendidas': 0}}